# 09 - Compare penalties and freeze the winner

Penalty selection uses only the highest unrounded mean SummaC of AlignScore-selected summaries. The baseline is a comparison, not a candidate. ROUGE-L is reported independently. 

## Fixed Configuration


In [1]:
from pathlib import Path
import json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
shared_notebook = ROOT / 'notebooks/05_prepare_validation_ablation.ipynb'
shared_cells = json.loads(shared_notebook.read_text(encoding='utf-8'))['cells']
for shared_index, shared_cell in enumerate(shared_cells):
    if shared_cell['cell_type'] == 'code' and 'ablation-shared' in shared_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(shared_cell['source']), str(shared_notebook) + ':cell' + str(shared_index), 'exec'), globals())
ablation = SummarizationAblation(CONFIG)
ablation.validate_run()


## Analysis implementation
All four methods require complete, ordered, hash-verified scores. Exact mean-SummaC ties retain the first configured penalty. The frozen configuration includes input identity, scoring configuration, software versions, and stage manifests.


In [2]:
def select_best_penalty(self):
    self.validate_run()
    table = []
    selected_scores = []
    for label in ['baseline', *[f'penalty_{p}' for p in self.c['penalties']]]:
        rows = self.load_completed_output(self.out / f'scores/{label}_scores.csv')
        for col in ['summac', 'rougeL']:
            valid_scores([float(r[col]) for r in rows], len(self.data))
        table.append(dict(method=label, average_summac=sum(float(r['summac']) for r in rows)/len(rows),
                          average_rougeL=sum(float(r['rougeL']) for r in rows)/len(rows)))
        if label != 'baseline':
            selected_scores.extend(dict(method=label, **r) for r in rows)
    ranked = sorted(table[1:], key=lambda r: -r['average_summac'])
    winner = float(ranked[0]['method'].removeprefix('penalty_'))
    save_csv(self.out / 'scores/selected_summaries_scores.csv', selected_scores)
    save_csv(self.out / 'analysis/ablation_comparison.csv', table)
    save_json(self.out / 'analysis/ablation_results.json', dict(results=table, ranked_penalties=ranked,
        selection_criterion='highest unrounded average SummaC of AlignScore-selected summaries',
        exact_tie_rule='first configured penalty; ROUGE-L never breaks ties'))
    frozen = dict(selected_diversity_penalty=winner, experiment=self.run_identity(),
                  progress_manifests={p.name: json.loads(p.read_text()) for p in sorted((self.out / 'progress').glob('*.json'))},
                  results=table, selection_criterion='average SummaC only',
                  candidate_tie_rule='first maximum AlignScore', final_full_pipeline_executed=False)
    frozen_path = self.out / 'best_diversity_penalty.json'
    if frozen_path.exists():
        require(json.loads(frozen_path.read_text()) == frozen, 'Frozen selection differs; refusing overwrite')
    save_json(frozen_path, frozen)
    report = '# Validation diversity penalty ablation\n\n'
    report += '619 official SciTLDR-A dev abstracts; stable source order. Five candidates per penalty, 10 beams, 5 groups, max_length=64.\n\n'
    report += 'AlignScore uses source abstracts and selects independently within each five-candidate group. SummaC evaluates selected summaries against abstracts. ROUGE-L F1 uses stemming and maximum over human references.\n\n'
    report += '| Method | Average SummaC | Average ROUGE-L |\n|---|---:|---:|\n'
    report += ''.join(f"| {r['method']} | {r['average_summac']:.8f} | {r['average_rougeL']:.8f} |\n" for r in table)
    report += f'\nSelected penalty: {winner}, exclusively by unrounded average SummaC. Exact ties retain the first configured penalty.\n\n'
    report += 'The 618-row test baseline is excluded from validation evaluation. The validation checkpoint and verified baseline source are recorded in the configuration. Full configuration, hashes, and software versions are in best_diversity_penalty.json. No final full-pipeline run was executed.\n'
    (self.out / 'analysis/ablation_report.md').write_text(report, encoding='utf-8')
    print(report)


SummarizationAblation.select_best_penalty = select_best_penalty


## Comparison, report, and frozen configuration
Outputs: analysis/ablation_comparison.csv, analysis/ablation_results.json, analysis/ablation_report.md, and best_diversity_penalty.json.


In [4]:
ablation.select_best_penalty()


# Validation diversity penalty ablation

619 official SciTLDR-A dev abstracts; stable source order. Five candidates per penalty, 10 beams, 5 groups, max_length=64.

AlignScore uses source abstracts and selects independently within each five-candidate group. SummaC evaluates selected summaries against abstracts. ROUGE-L F1 uses stemming and maximum over human references.

| Method | Average SummaC | Average ROUGE-L |
|---|---:|---:|
| baseline | 0.77342567 | 0.34987846 |
| penalty_0.5 | 0.86784349 | 0.35032923 |
| penalty_1.0 | 0.89476536 | 0.34645641 |
| penalty_2.0 | 0.85775008 | 0.34710678 |

Selected penalty: 1.0, exclusively by unrounded average SummaC. Exact ties retain the first configured penalty.

The 618-row test baseline is excluded from validation evaluation. The validation checkpoint and verified baseline source are recorded in the configuration. Full configuration, hashes, and software versions are in best_diversity_penalty.json. No final full-pipeline run was executed.



## Frozen selection
The study ends with the selected configuration.


In [6]:
selected = json.loads((ablation.out / 'best_diversity_penalty.json').read_text(encoding='utf-8'))
print('Selected diversity penalty:', selected['selected_diversity_penalty'])
print('Criterion:', selected['selection_criterion'])


Selected diversity penalty: 1.0
Criterion: average SummaC only
